# Layered 16-Qubit Entanglement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


# ============================================================
# 1. Definition

n_qubits = 16
grid_size = 4

ry_angles = np.linspace(np.pi / 10, np.pi / 3, n_qubits)
rz_angles = np.linspace(np.pi / 12, np.pi / 5, n_qubits)

qc = QuantumCircuit(n_qubits)


# =======================================
# 2. Hadamard

for q in range(n_qubits):
    qc.h(q)

qc.barrier()


# ======================================
# 3. Parameterized Single-Qubit Layer

for q in range(n_qubits):
    qc.ry(ry_angles[q], q)
    qc.rz(rz_angles[q], q)

qc.barrier()


# ======================================
# 4. Horizontal Nearest-Neighbor CX Layer

horizontal_edges = []

for row in range(grid_size):
    start = row * grid_size

    for col in range(grid_size - 1):
        q1 = start + col
        q2 = start + col + 1

        horizontal_edges.append((q1, q2))
        qc.cx(q1, q2)

qc.barrier()


# ======================================
# 5. Vertical CX and Diagonal RZZ Layer

vertical_edges = []
diagonal_edges = []

for row in range(grid_size - 1):
    for col in range(grid_size):

        q1 = row * grid_size + col
        q2 = (row + 1) * grid_size + col

        vertical_edges.append((q1, q2))
        qc.cx(q1, q2)

        if col < grid_size - 1:
            q_dr = (row + 1) * grid_size + col + 1

            diagonal_edges.append((q1, q_dr))
            qc.rzz(np.pi / 6, q1, q_dr)

        if col > 0:
            q_dl = (row + 1) * grid_size + col - 1

            diagonal_edges.append((q1, q_dl))
            qc.rzz(np.pi / 7, q1, q_dl)

qc.barrier()

long_range_edges = [
    (0, 15),
    (3, 12),
    (1, 14),
    (2, 13),
    (4, 11),
    (7, 8),
]

for q1, q2 in long_range_edges:
    qc.rzz(np.pi / 8, q1, q2)

qc.barrier()

for row in range(grid_size):
    start = row * grid_size

    for col in reversed(range(grid_size - 1)):
        q1 = start + col + 1
        q2 = start + col

        qc.cx(q1, q2)

qc.barrier()


# ======================================
# 6. Parameterized Single-Qubit Layer

for q in range(n_qubits):
    qc.ry(np.pi / 7, q)
    qc.rz(np.pi / 9, q)


# ======================================
# 7. Display

print("Geometric Entangling Circuit")
print("=" * 60)
print(qc.draw(output="text"))


# ======================================
# 8. Statevector

statevector = Statevector.from_instruction(qc)

print("\nStatevector Analysis")
print("=" * 60)
print(f"# of qubits: {n_qubits}")
print(f"Statevector dimension: {len(statevector.data):,}")
print(f"Expected dimension: {2 ** n_qubits:,}")

probabilities = np.abs(statevector.data) ** 2

basis_states = [
    format(index, f"0{n_qubits}b")[::-1]
    for index in range(len(probabilities))
]

sorted_indices = np.argsort(probabilities)[::-1]

print("\nTop 10 Computational-Basis States")
print("=" * 60)

for rank, index in enumerate(sorted_indices[:10], start=1):
    print(
        f"{rank:2d}. |{basis_states[index]}> "
        f"-> Probability = {probabilities[index]:.6f}"
    )


# ======================================
# 9. Visualization

top_indices = sorted_indices[:10]

top_states = [
    basis_states[index]
    for index in top_indices
]

top_probabilities = [
    probabilities[index]
    for index in top_indices
]

plt.figure(figsize=(10, 6))

y_pos = np.arange(len(top_states))

plt.barh(
    y_pos,
    top_probabilities[::-1],
    color='teal'
)

plt.yticks(
    y_pos,
    [f"|{state}>" for state in top_states[::-1]],
    fontfamily="monospace"
)

plt.xlabel("Probability")
plt.ylabel("Computational-Basis State (|q0...q15>)")
plt.title("Top 10 States")

plt.tight_layout()
plt.show()


# ======================================
# 10. Pairwise ZZ Correlation Matrix

indices = np.arange(2 ** n_qubits)

bit_matrix = (
    (indices[:, None] & (1 << np.arange(n_qubits))) > 0
).astype(np.int8)

z_values = 1 - 2 * bit_matrix

correlation_matrix = (
    z_values.T * probabilities
) @ z_values


# ======================================
# 11. Correlation Visualization

plt.figure(figsize=(8, 7))

plt.imshow(
    correlation_matrix,
    interpolation="nearest",
    aspect="auto"
)

plt.colorbar(
    label=r"$\langle Z_i Z_j \rangle$"
)

plt.xticks(range(n_qubits))
plt.yticks(range(n_qubits))

plt.xlabel("Qubit index")
plt.ylabel("Qubit index")
plt.title("Pairwise ZZ Correlation Matrix")

plt.tight_layout()
plt.show()

# QAOA Parameter Landscape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp


### 1. Definition

n_qubits = 3

def qubo_objective(x):
    x0, x1, x2 = x

    return (x0+ x1+ x2- 2 * x0 * x1- 2 * x1 * x2+ x0 * x2)

bitstrings = list(product([0, 1], repeat=n_qubits))

classical_values = {x: qubo_objective(x)
    for x in bitstrings
}

classical_optimum = min(classical_values.values())

classical_solutions = [x
    for x, value in classical_values.items()
    if np.isclose(value, classical_optimum)
]

for x, value in classical_values.items():
    print(f"{x} -> Objective = {value:.1f}")

print(f"\nClassical optimum = {classical_optimum:.1f}")
print(f"Optimal binary solutions = {classical_solutions}")

### 2. Convert QUBO --> Ising Hamiltonian

cost_hamiltonian = SparsePauliOp.from_list([
    ("III",  0.75),
    ("IIZ", -0.25),
    ("IZI",  0.50),
    ("ZII", -0.25),
    ("IZZ", -0.50),
    ("ZZI", -0.50),
    ("ZIZ",  0.25)
])


print("\nQiskit Ising Hamiltonian")
print(cost_hamiltonian)
print("#" * 30)

### 3. Build QAOA

def apply_cost_layer(qc, hamiltonian, gamma):

    for pauli, coefficient in zip(
        hamiltonian.paulis,
        hamiltonian.coeffs
    ):
        coefficient = float(np.real(coefficient))
        pauli_string = pauli.to_label()

        if "Z" not in pauli_string:
            continue

        z_qubits = [
            qubit
            for qubit, operator in enumerate(
                reversed(pauli_string)
            )
            if operator == "Z"
        ]

        if len(z_qubits) == 1:
            qc.rz(
                2 * gamma * coefficient,
                z_qubits[0]
            )

        elif len(z_qubits) == 2:
            qc.rzz(
                2 * gamma * coefficient,
                z_qubits[0],
                z_qubits[1]
            )

        else:
            raise ValueError(
                f"Unsupported Pauli term: {pauli_string}"
            )


## 4. Circuit

def qaoa_state(beta, gamma):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))

    apply_cost_layer(
        qc,
        cost_hamiltonian,
        gamma
    )

    qc.rx(
        2 * beta,
        range(n_qubits)
    )

    return Statevector.from_instruction(qc)

### 5. QAOA Expectation Value

def expectation_value(params):
    beta, gamma = params
    state = qaoa_state(beta,gamma)

    return float(np.real(state.expectation_value(cost_hamiltonian)))

### 6. QAOA Expected Energy landscape

beta_values = np.linspace(0,np.pi,81)
gamma_values = np.linspace(-np.pi,np.pi,81)
energy_landscape = np.zeros((len(beta_values),len(gamma_values)))


for i, beta in enumerate(beta_values):

    for j, gamma in enumerate(gamma_values):
        energy_landscape[i, j] = expectation_value(
            [beta, gamma]
        )

grid_min_index = np.unravel_index(
    np.argmin(energy_landscape),
    energy_landscape.shape
)

grid_beta = beta_values[grid_min_index[0]]
grid_gamma = gamma_values[grid_min_index[1]]
grid_energy = energy_landscape[grid_min_index]

print(f"Best grid beta  = {grid_beta:.6f}")
print(f"Best grid gamma = {grid_gamma:.6f}")
print(f"Grid minimum energy = {grid_energy:.6f}")

### 7. Visualization QAOA

plt.figure(figsize=(9, 6))

plt.imshow(
    energy_landscape,
    origin="lower",
    aspect="auto",
    extent=[
        gamma_values[0],
        gamma_values[-1],
        beta_values[0],
        beta_values[-1]
    ],
    interpolation="nearest"
)

plt.colorbar(
    label="Expected energy"
)

plt.scatter(
    [grid_gamma],
    [grid_beta],
    s=100,
    marker="x",
    linewidths=3,
    label="Best grid point"
)

plt.xlabel(r"$\gamma$")
plt.ylabel(r"$\beta$")
plt.title("p=1 QAOA Energy Landscape")

plt.legend()
plt.tight_layout()
plt.show()


### 8. Local Numeric

result = minimize(expectation_value,
    x0=np.array([
        grid_beta,
        grid_gamma
    ]),
    method="COBYLA",
    options={
        "maxiter": 300
    }
)

optimal_beta, optimal_gamma = result.x
optimal_energy = expectation_value(result.x)

print("\nQAOA optimization")
print(f"Optimal beta  = {optimal_beta:.6f}")
print(f"Optimal gamma = {optimal_gamma:.6f}")
print(f"Minimum expected energy = {optimal_energy:.6f}")

### 9. Final QAOA Probability

optimized_state = qaoa_state(optimal_beta,optimal_gamma)
probabilities = optimized_state.probabilities()
ranked_indices = np.argsort(probabilities)[::-1]

print("\nFinal QAOA probability distribution")
print("#" * 20)

for index in ranked_indices:
    probability = probabilities[index]
    if probability < 1e-6:
        continue
    qiskit_bitstring = format(
        index,
        f"0{n_qubits}b"
    )

    logical_bitstring = qiskit_bitstring[::-1]
    logical_bits = tuple(
        map(int, logical_bitstring)
    )

    objective = qubo_objective(
        logical_bits
    )
    print(
        f"|{qiskit_bitstring}> -> "
        f"Probability = {probability:.6f} | "
        f"QUBO = {objective:.1f}"
    )

# Weighted MaxCut Application

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp


### 1. Definition

n_qubits = 6

weighted_edges = [
    (0, 1, 1.0),
    (0, 2, 2.0),
    (1, 2, 1.5),
    (1, 3, 2.5),
    (2, 4, 2.0),
    (3, 4, 1.0),
    (3, 5, 2.5),
    (4, 5, 1.5),
    (2, 5, 1.0)
]


### 2. Visualize the weighted graph

graph = nx.Graph()

graph.add_nodes_from(range(n_qubits))

for i, j, weight in weighted_edges:
    graph.add_edge(i,j,weight=weight)

positions = nx.spring_layout(graph,seed=42)

plt.figure(figsize=(9, 7))

nx.draw_networkx_nodes(
    graph,
    positions,
    node_size=900
)

nx.draw_networkx_edges(
    graph,
    positions,
    width=[
        1.5 + weight
        for _, _, weight in weighted_edges
    ]
)

nx.draw_networkx_labels(
    graph,
    positions,
    font_size=11
)

edge_labels = {
    (i, j): f"{weight:.1f}"
    for i, j, weight in weighted_edges
}

nx.draw_networkx_edge_labels(
    graph,
    positions,
    edge_labels=edge_labels,
    font_size=9
)

plt.title("Initial MaxCut Graph")
plt.axis("off")
plt.tight_layout()
plt.show()


### 3. Classical MaxCut

def weighted_cut_value(bitstring):
    return sum(weight
        for i, j, weight in weighted_edges
        if bitstring[i] != bitstring[j]
    )


all_bitstrings = list(
    product([0, 1], repeat=n_qubits)
)

classical_values = {
    bitstring: weighted_cut_value(bitstring)
    for bitstring in all_bitstrings
}

classical_optimum = max(
    classical_values.values()
)

classical_solutions = [
    bitstring
    for bitstring, value in classical_values.items()
    if np.isclose(value,classical_optimum)
]

total_edge_weight = sum(
    weight
    for _, _, weight in weighted_edges
)


print("Classical Weighted MaxCut")
print("-" * 30)

print(
    f"Total edge weight = "
    f"{total_edge_weight:.1f}"
)

print(
    f"Classical maximum cut = "
    f"{classical_optimum:.1f}"
)

print(
    f"Number of optimal solutions = "
    f"{len(classical_solutions)}"
)

print(
    f"Optimal solutions = "
    f"{classical_solutions}"
)


### 4. Construct Ising Hamiltonian

pauli_terms = []

for i, j, weight in weighted_edges:

    pauli = ["I"] * n_qubits

    pauli[i] = "Z"
    pauli[j] = "Z"

    pauli_string = "".join(
        pauli[::-1]
    )

    pauli_terms.append(
        (
            pauli_string,
            weight / 2
        )
    )

cost_hamiltonian = SparsePauliOp.from_list(
    pauli_terms
)


print("\nWeighted Ising Hamiltonian")
print(cost_hamiltonian)


### 5. Apply QAOA

def apply_cost_layer(
    qc,
    hamiltonian,
    gamma
):

    for pauli, coefficient in zip(
        hamiltonian.paulis,
        hamiltonian.coeffs
    ):

        coefficient = float(
            np.real(coefficient)
        )

        pauli_string = pauli.to_label()

        z_qubits = [
            qubit
            for qubit, operator
            in enumerate(
                reversed(pauli_string)
            )
            if operator == "Z"
        ]

        if len(z_qubits) == 2:

            qc.rzz(
                2 * gamma * coefficient,
                z_qubits[0],
                z_qubits[1]
            )

        elif len(z_qubits) != 0:

            raise ValueError(
                f"Unsupported Pauli term: "
                f"{pauli_string}"
            )


### 6. QAOA State

def qaoa_state(params):

    beta_1, beta_2, gamma_1, gamma_2 = params

    qc = QuantumCircuit(n_qubits)

    qc.h(range(n_qubits))

    apply_cost_layer(
        qc,
        cost_hamiltonian,
        gamma_1
    )

    qc.rx(
        2 * beta_1,
        range(n_qubits)
    )

    apply_cost_layer(
        qc,
        cost_hamiltonian,
        gamma_2
    )

    qc.rx(
        2 * beta_2,
        range(n_qubits)
    )

    return Statevector.from_instruction(qc)


### 7. Expected Value

def expectation_value(params):

    state = qaoa_state(params)

    return float(
        np.real(
            state.expectation_value(
                cost_hamiltonian
            )
        )
    )


### 8. Multi-start Parameter Optimization

initial_points = [
    np.array([0.20, 0.40, 0.30, 0.70]),
    np.array([0.60, 0.20, 0.80, 0.40]),
    np.array([0.90, 0.70, 0.50, 0.20]),
    np.array([0.25, 0.85, 0.55, 0.75]),
    np.array([0.55, 0.50, 0.55, 0.85]),
    np.array([0.15, 0.75, 0.25, 0.45]),
    np.array([0.10, 0.80, 0.55, 0.45])
]

results = []

for initial_point in initial_points:

    result = minimize(
        expectation_value,
        initial_point,
        method="COBYLA",
        options={
            "maxiter": 400
        }
    )

    results.append(result)


best_result = min(
    results,
    key=lambda result: result.fun
)

optimal_params = best_result.x
optimal_energy = best_result.fun

estimated_cut = (
    total_edge_weight / 2
    - optimal_energy
)


print("\nQAOA p=2 optimization")
print("-" * 30)

print(f"beta_1  = {optimal_params[0]:.6f}")

print(f"beta_2  = {optimal_params[1]:.6f}")

print(f"gamma_1 = {optimal_params[2]:.6f}")

print(f"gamma_2 = {optimal_params[3]:.6f}")

print(
    f"Minimum expected energy = "
    f"{optimal_energy:.6f}"
)

print(
    f"Expected weighted cut = "
    f"{estimated_cut:.6f}"
)

### 9. QAOA Circuit Visualization

optimized_circuit = QuantumCircuit(n_qubits)
optimized_circuit.h(range(n_qubits))
apply_cost_layer(optimized_circuit,cost_hamiltonian,optimal_params[2])
optimized_circuit.rx(2 * optimal_params[0],range(n_qubits))
apply_cost_layer(optimized_circuit,cost_hamiltonian,optimal_params[3])
optimized_circuit.rx(2 * optimal_params[1],range(n_qubits))
print(optimized_circuit.draw(fold=140))

### 10. Quantum probability

optimized_state = qaoa_state(optimal_params)
probabilities = optimized_state.probabilities()
ranked_indices = np.argsort(probabilities)[::-1]

top_k = 12
top_states = []

for index in ranked_indices[:top_k]:

    qiskit_bitstring = format(index,f"0{n_qubits}b")

    logical_bitstring = qiskit_bitstring[::-1]

    logical_bits = tuple(
        map(int,logical_bitstring))

    top_states.append(
        ("".join(
                map(str,logical_bits)),
            probabilities[index],
            weighted_cut_value(
                logical_bits
            )
        )
    )


print("\nTop QAOA states")
print("-" * 30)

for bitstring, probability, cut in top_states:

    print(
        f"|{bitstring}> -> "
        f"Probability = {probability:.6f} | "
        f"Cut = {cut:.1f}"
    )


## 11. Visualize the probability (Horizontal Bar Chart)

labels = [state[0] for state in top_states]
values = [state[1] for state in top_states]
cuts = [state[2] for state in top_states]

plt.figure(figsize=(10, 6))

bars = plt.barh(labels, values, color="teal")

for bar, cut in zip(bars, cuts):
  plt.text(
      bar.get_width() + 0.005, 
      bar.get_y() + bar.get_height() / 2,
      f"cut = {cut:.1f}",
      ha="left",
      va="center",
      fontsize=9,
  )

plt.xlabel("Probability")
plt.ylabel("Computational basis state")
plt.title("Top QAOA States and Their Weighted Cut Values")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


### 12. Best QAOA Sol.

best_qaoa_index = max(
    range(len(probabilities)),
    key=lambda index: weighted_cut_value(
        tuple(
            map(int,format(index, f"0{n_qubits}b")[::-1])
        )
    )
)

best_qaoa_bitstring = format(
    best_qaoa_index,
    f"0{n_qubits}b"
)[::-1]

best_qaoa_state = tuple(map(int, best_qaoa_bitstring))
best_qaoa_cut = weighted_cut_value(best_qaoa_state)

print("\nBest QAOA solution")
print("#" * 30)

print(f"Best QAOA state = {best_qaoa_state}")
print(f"Weighted cut value = {best_qaoa_cut:.1f}")
print(f"Classical optimum = {classical_optimum:.1f}")


### 13. Visualize the QAOA partition
partition_a = [
    q
    for q, value in enumerate(best_qaoa_state)
    if value == 0]

partition_b = [
    q
    for q, value in enumerate(best_qaoa_state)
    if value == 1]

node_colors = [
    "#1f77b4" if node in partition_a else "#ff7f0e"
    for node in graph.nodes()]

node_shapes = {
    0: "o",
    1: "s"}
plt.figure(figsize=(9, 7))

nx.draw_networkx_nodes(
    graph,
    positions,
    nodelist=partition_a,
    node_color="#1f77b4",
    node_size=950,
    node_shape=node_shapes[0]
)

nx.draw_networkx_nodes(
    graph,
    positions,
    nodelist=partition_b,
    node_color="#ff7f0e",
    node_size=950,
    node_shape=node_shapes[1]
)

nx.draw_networkx_edges(
    graph,
    positions,
    width=[
        1.5 + weight
        for _, _, weight in weighted_edges]
)

nx.draw_networkx_labels(
    graph,
    positions,
    font_size=11
)

nx.draw_networkx_edge_labels(
    graph,
    positions,
    edge_labels=edge_labels,
    font_size=9
)

plt.title(
    "QAOA Maximum-Cut Partition\n"
    f"Cut value = {best_qaoa_cut:.1f}"
)

plt.axis("off")
plt.tight_layout()
plt.show()


### 14. Edge-by-edge contribution

cut_edges = [
    (
        i,
        j,
        weight
    )
    for i, j, weight in weighted_edges
    if best_qaoa_state[i]
    != best_qaoa_state[j]
]

print("\nEdges crossing the selected cut")
print("#" * 30)

for i, j, weight in cut_edges:

    print(
        f"({i}, {j}) -> "
        f"weight = {weight:.1f}"
    )

print(
    f"\nSum == "
    f"{sum(weight for _, _, weight in cut_edges):.1f}"
)